# Two facing PET modules
Measured PE signals and first/second-interaction MC truth. Start by changing `CSV`.
Local depth runs from the glass face (0 mm) to the SiPM end (crystal length).
This baseline does **not** model polarization entanglement or random coincidences.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = Path('length-15/events.csv')
df = pd.read_csv(CSV)
print(f'{len(df):,} generated events; {df.both_detected.sum():,} detected in both modules')
df[['EventID', 'A_npe_total', 'B_npe_total', 'A_truth_i1_type', 'B_truth_i1_type']].head()


Select detected events. Keep the full dataframe so cuts can be changed without rereading it. The detection condition is same-event detection, not a timing-window selection.

In [ ]:
selected = df.loc[df.both_detected].copy()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, module in zip(axes, ['A', 'B']):
    ax.hist(selected[f'{module}_npe_total'], bins=60, histtype='stepfilled', alpha=0.65)
    ax.set(xlabel=f'Module {module}: total PE', ylabel='Events')
fig.tight_layout()
plt.show()


DOI study: light-sharing ratio versus the true first-interaction depth. PE counts are not yet energy calibrated.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, module in zip(axes, ['A', 'B']):
    points = selected[[f'{module}_truth_i1_depth_mm', f'{module}_w']].dropna()
    ax.scatter(points.iloc[:, 0], points.iloc[:, 1], s=3, alpha=0.2)
    ax.set(xlabel=f'Module {module}: true depth (mm)', ylabel='W = max PE / total PE')
fig.tight_layout()
plt.show()


Scattering study: count first/second interaction combinations, then require CS followed by PE on the selected track. A missing second interaction remains missing.

In [ ]:
display(pd.crosstab(selected.A_truth_i1_type.fillna('missing'),
                    selected.A_truth_i2_type.fillna('missing')))
scatters = selected.loc[
    selected.truth_distinct_pair
    & selected.A_truth_i1_type.eq('CS')
    & selected.A_truth_i2_type.eq('PE')
].copy()
print(f'{len(scatters):,} events with module A CS then PE')
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(scatters.A_truth_compton_angle_deg, scatters.A_truth_two_site_angle_deg, s=5, alpha=0.3)
ax.plot([0, 180], [0, 180], '--', color='black')
ax.set(xlabel='True first-scatter angle (degrees)', ylabel='Two-site angle (degrees)')
fig.tight_layout()
plt.show()


The two-module scattering-plane angle is truth-level geometry. Its distribution here is a baseline, not an entanglement prediction.

In [ ]:
double_cs = selected.loc[selected.truth_distinct_pair
                         & selected.A_truth_i1_type.eq('CS')
                         & selected.B_truth_i1_type.eq('CS')]
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(double_cs.truth_plane_angle_deg.dropna(), bins=np.linspace(0, 90, 19), histtype='step')
ax.set(xlabel='Folded scattering-plane separation (degrees)', ylabel='Events')
fig.tight_layout()
plt.show()
